# LambForce-EC Step 3 production workflow
This notebook installs the package from GitHub, mounts Google Drive, creates a unique timestamped runtime directory, validates the package, and runs a non-claim-bearing synthetic software test. Replace the synthetic NPZ only with a verified hydrodynamic archive record and checksum for scientific runs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
base = Path('/content/drive/MyDrive/LambForce-EC/runtime')
run_dir = base / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_dir.mkdir(parents=True, exist_ok=False)
print(run_dir)

In [ ]:
!python -m pip install --upgrade pip -q
!python -m pip install 'git+https://github.com/khalid-saqr/LambForce-EC.git@agent/step3-production-package' -q

In [ ]:
from lambforce_ec.synthetic import make_synthetic_artery
from lambforce_ec.io import save_artery_npz, save_result
from lambforce_ec.workflow import run_case, run_registered_comparison
from lambforce_ec.models import RunConfig
record = make_synthetic_artery()
save_artery_npz(record, run_dir / 'synthetic_input.npz')
config = RunConfig(load_distribution='localized_bound')
result = run_case(record, config=config)
save_result(result, run_dir / 'synthetic_result')
comparison = run_registered_comparison(record, config=config)
print(result.metadata['configuration_checksum'])
print(comparison['incremental_lamb_wss_ratio'])

## Claim-bearing use
Upload or mount a verified artery NPZ conforming to `schemas/artery_input.schema.json`, verify its source checksum, load `configs/step3_default.yaml`, and run the registered comparison. Do not treat the synthetic result as biological evidence.